# Import libraries

In [9]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI,GoogleGenerativeAIEmbeddings
from langchain_classic.retrievers import BM25Retriever, EnsembleRetriever
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader

load_dotenv()

True

# Initialize vectorstore and retrievers

In [3]:
api_key = os.getenv("API_KEY")
llm = model = ChatGoogleGenerativeAI(
    api_key=api_key,
    model="gemini-2.5-flash-lite",
    temperature=0.3,
    max_tokens=50000,
    timeout=None,
    max_retries=2
)

collection_name = "langchain_docs_index"
namespace = f"chroma/{collection_name}"
embedding = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001", api_key=api_key)

vectorstore = Chroma(embedding_function=embedding, collection_name=collection_name, persist_directory="./data/vectors/chroma_db")

In [10]:
data_dir = "data"
loader = PyPDFLoader(os.path.join(data_dir, "book_chapter_02.pdf"))
docs = loader.load()

In [12]:
bm25_retriever = BM25Retriever.from_documents(docs)
bm25_retriever.k = 2

In [13]:
vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# Ensemble retriever

In [14]:
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever], weights=[0.5, 0.5]
)

In [15]:
ensemble_retriever.invoke("What are Morphemes?")

[Document(metadata={'producer': 'PDFium', 'creator': 'PDFium', 'creationdate': 'D:20260329110007', 'source': 'data\\book_chapter_02.pdf', 'total_pages': 34, 'page': 15, 'page_label': '16'}, page_content='2.6 • R EGULAR EXPRESSIONS 19\nLanguage variety: What language (including dialect/region) was the corpus in?\nSpeaker demographics: What was, e.g., the age or gender of the text’s authors?\nCollection process: How big is the data? If it is a subsample how was it sampled?\nWas the data collected with consent? How was the data pre-processed, and\nwhat metadata is available?\nAnnotation process: What are the annotations, what are the demographics of the\nannotators, how were they trained, how was the data annotated?\nDistribution: Are there copyright or other intellectual property restrictions?\n2.6 Regular Expressions\nOne of the most useful tools for text processing in computer science is the regular\nexpression (or regex), a language for specifying text strings. Regexes are used inregu